In [0]:
# Comparer le nombre de lignes Bronze et Silver
# Definir les tables a controler
tables = ["customers", "products", "sessions", "transactions", "reviews"]
for table in tables:
    bronze = spark.table(f"`E-commerce`.bronze.{table}")
    silver = spark.table(f"`E-commerce`.silver.{table}")

    print(
        table,
        "| Bronze:", bronze.count(),
        "| Silver:", silver.count()
    )

In [0]:
# Compter les doublons dans Silver
for table in tables:
    df = spark.table(f"`E-commerce`.silver.{table}")
    doublons = df.count() - df.dropDuplicates().count()

    print(table, ":", doublons, "doublons")

In [0]:
# Definir les cles principales
primary_keys = {
    "customers": "customer_id",
    "products": "product_id",
    "sessions": "session_id",
    "transactions": "transaction_id",
    "reviews": "review_id"
}

In [0]:
# Verifier les ID dupliques
for table, key in primary_keys.items():
    df = spark.table(f"`E-commerce`.silver.{table}")

    doublons_id = (
        df.groupBy(key)
        .count()
        .filter("count > 1")
        .count()
    )

    print(table, ":", doublons_id, "ID dupliques")

In [0]:
# Verifier les ID NULL
from pyspark.sql.functions import col

for table, key in primary_keys.items():
    df = spark.table(f"`E-commerce`.silver.{table}")
    null_id = df.filter(col(key).isNull()).count()

    print(table, ":", null_id, "ID NULL")

In [0]:
# Verifier les valeurs negatives des transactions
transactions = spark.table("`E-commerce`.silver.transactions")

invalid_transactions = transactions.filter(
    (col("quantity") < 0) |
    (col("unit_price") < 0)
).count()

print("Transactions invalides :", invalid_transactions)

In [0]:
# Verifier le calcul du montant total
invalid_amounts = transactions.filter(
    col("total_amount") != col("quantity") * col("unit_price")
).count()

print("Montants incoherents :", invalid_amounts)

In [0]:
# Verifier les ratings invalides
reviews = spark.table("`E-commerce`.silver.reviews")

invalid_ratings = reviews.filter(
    (col("rating") < 1) |
    (col("rating") > 5)
).count()

print("Ratings invalides :", invalid_ratings)

In [0]:
# Verifier les valeurs negatives des sessions
sessions = spark.table("`E-commerce`.silver.sessions")

invalid_sessions = sessions.filter(
    (col("duration_seconds") < 0) |
    (col("pages_viewed") < 0)
).count()

print("Sessions invalides :", invalid_sessions)